In [2]:
import geopandas as gpd
import pandas as pd
import requests
from shapely.geometry import Point
from tqdm import tqdm

# ==============================
# INPUTS
# ==============================
AOI_SHP = "./shapefiles/kakamega_area_001.shp"
OUTPUT_CSV = "Kakamega_butterflies_inaturalist.csv"

# Lepidoptera (butterflies + moths)
TAXON_ID = 47224

# ==============================
# LOAD AOI
# ==============================
print("Loading AOI...")

aoi = gpd.read_file(AOI_SHP)

# Convert to WGS84
if aoi.crs != "EPSG:4326":
    aoi = aoi.to_crs("EPSG:4326")

# Get bounding box
xmin, ymin, xmax, ymax = aoi.total_bounds

# ==============================
# DOWNLOAD OBSERVATIONS
# ==============================
print("Downloading observations from iNaturalist...")

base_url = "https://api.inaturalist.org/v1/observations"

all_records = []
page = 1
per_page = 200

while True:

    params = {
        "taxon_id": TAXON_ID,
        "nelat": ymax,
        "nelng": xmax,
        "swlat": ymin,
        "swlng": xmin,
        "quality_grade": "research",
        "per_page": per_page,
        "page": page,
        "order_by": "observed_on"
    }

    r = requests.get(base_url, params=params, timeout=60)
    r.raise_for_status()

    data = r.json()

    results = data["results"]

    if len(results) == 0:
        break

    for obs in results:

        coords = obs.get("geojson", {}).get("coordinates")

        if coords is None:
            continue

        lon, lat = coords

        taxon = obs.get("taxon")

        species = None
        family = None

        if taxon:
            species = taxon.get("name")

            for anc in taxon.get("ancestors", []):
                if anc.get("rank") == "family":
                    family = anc.get("name")

        all_records.append({
            "species": species,
            "family": family,
            "longitude": lon,
            "latitude": lat
        })

    print(f"Page {page}: {len(results)} records")

    if len(results) < per_page:
        break

    page += 1

print(f"\nDownloaded {len(all_records):,} observations")

# ==============================
# CREATE GEODATAFRAME
# ==============================
df = pd.DataFrame(all_records)

gdf = gpd.GeoDataFrame(
    df,
    geometry=gpd.points_from_xy(df.longitude, df.latitude),
    crs="EPSG:4326"
)

# ==============================
# FILTER TO AOI
# ==============================
print("Filtering to AOI boundary...")

gdf = gpd.sjoin(
    gdf,
    aoi[["geometry"]],
    predicate="within",
    how="inner"
)

# ==============================
# KEEP BUTTERFLIES ONLY
# ==============================
butterfly_families = [
    "Papilionidae",
    "Pieridae",
    "Nymphalidae",
    "Lycaenidae",
    "Riodinidae",
    "Hesperiidae"
]

gdf = gdf[gdf["family"].isin(butterfly_families)]

# ==============================
# EXPORT
# ==============================
out = (
    gdf[["species", "longitude", "latitude"]]
    .drop_duplicates()
    .reset_index(drop=True)
)

out.to_csv(OUTPUT_CSV, index=False)

print(f"Saved {len(out):,} butterfly records")
print(f"Output: {OUTPUT_CSV}")

Loading AOI...
Page 1: 200 records
Page 2: 200 records
Page 3: 200 records
Page 4: 129 records

Downloaded 729 observations
Filtering to AOI boundary...
Saved 0 butterfly records
Output: Kakamega_butterflies_inaturalist.csv


In [3]:
import geopandas as gpd
import pandas as pd
import requests
import time

def extract_butterflies_from_shapefile(shapefile_path: str, output_csv: str = "butterflies.csv"):
    """
    Extracts butterfly observations from iNaturalist within a shapefile's boundary
    and exports longitude, latitude, and species name to CSV.
    """
    # 1. Load shapefile and reproject to WGS84 (EPSG:4326) for lat/lon matching
    print("Loading shapefile...")
    gdf = gpd.read_file(shapefile_path)
    if gdf.crs is None or gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)

    # Combine multi-polygon shapefile features into a single geometry
    try:
        aoi_geometry = gdf.geometry.union_all()
    except AttributeError:
        # Backwards compatibility for older GeoPandas/Shapely versions
        aoi_geometry = gdf.geometry.unary_union

    # 2. Get bounding box coordinates for initial iNaturalist API query
    min_x, min_y, max_x, max_y = gdf.total_bounds

    # 3. Query iNaturalist API
    # Taxon ID 47224 = Superfamily Papilionoidea (Butterflies)
    url = "https://api.inaturalist.org/v1/observations"
    params = {
        "taxon_id": 47224,
        "swlat": min_y,
        "swlng": min_x,
        "nelat": max_y,
        "nelng": max_x,
        "per_page": 200,
        "verifiable": "true",  # Only fetch observations with valid, verifiable data
        "page": 1
    }

    observations = []
    print("Fetching observations from iNaturalist API...")

    while True:
        response = requests.get(url, params=params)
        if response.status_code != 200:
            print(f"API Error on page {params['page']}: HTTP {response.status_code}")
            break

        data = response.json()
        results = data.get("results", [])
        if not results:
            break

        for obs in results:
            location = obs.get("location")
            taxon = obs.get("taxon")

            if location and taxon:
                lat_str, lng_str = location.split(",")
                species_name = taxon.get("name")
                common_name = taxon.get("preferred_common_name", "")

                if species_name:
                    observations.append({
                        "longitude": float(lng_str),
                        "latitude": float(lat_str),
                        "species_name": species_name,
                        "common_name": common_name
                    })

        total_results = data.get("total_results", 0)
        if params["page"] * params["per_page"] >= total_results:
            break

        params["page"] += 1
        time.sleep(0.3)  # Respect API rate limits

    if not observations:
        print("No butterfly observations found in the specified area.")
        return

    # 4. Perform spatial filter (Point-in-Polygon)
    print("Filtering points inside exact shapefile geometry...")
    obs_df = pd.DataFrame(observations)
    obs_gdf = gpd.GeoDataFrame(
        obs_df,
        geometry=gpd.points_from_xy(obs_df.longitude, obs_df.latitude),
        crs="EPSG:4326"
    )

    # Filter observations strictly inside the polygon boundary
    inside_aoi = obs_gdf[obs_gdf.geometry.within(aoi_geometry)]

    # 5. Save to CSV
    output_df = inside_aoi[["longitude", "latitude", "species_name", "common_name"]]
    output_df.to_csv(output_csv, index=False)
    print(f"Success! Saved {len(output_df)} butterfly observations to '{output_csv}'.")


# --- Example Execution ---
if __name__ == "__main__":
    # Replace with the path to your shapefile
    SHAPEFILE_PATH = "./shapefiles/kakamega_area_001.shp"
    
    extract_butterflies_from_shapefile(SHAPEFILE_PATH, "butterflies_aoi.csv")

Loading shapefile...
Fetching observations from iNaturalist API...
Filtering points inside exact shapefile geometry...
Success! Saved 1250 butterfly observations to 'butterflies_aoi.csv'.


In [5]:
import time
import geopandas as gpd
import pandas as pd
import requests
from pygbif import occurrences as gbif_occ


def fetch_inaturalist_butterflies(min_x, min_y, max_x, max_y):
    """Fetches butterfly observations from iNaturalist API using bounding box."""
    print("Fetching from iNaturalist...")
    url = "https://api.inaturalist.org/v1/observations"
    params = {
        "taxon_id": 47224,  # Superfamily Papilionoidea (Butterflies)
        "swlat": min_y,
        "swlng": min_x,
        "nelat": max_y,
        "nelng": max_x,
        "per_page": 200,
        "verifiable": "true",
        "page": 1,
    }

    records = []
    while True:
        response = requests.get(url, params=params)
        if response.status_code != 200:
            print(f"iNaturalist API error: HTTP {response.status_code}")
            break

        data = response.json()
        results = data.get("results", [])
        if not results:
            break

        for obs in results:
            location = obs.get("location")
            taxon = obs.get("taxon")
            if location and taxon and taxon.get("name"):
                lat_str, lng_str = location.split(",")
                records.append({
                    "longitude": float(lng_str),
                    "latitude": float(lat_str),
                    "species_name": taxon.get("name"),
                    "source": "iNaturalist",
                })

        if params["page"] * params["per_page"] >= data.get("total_results", 0):
            break

        params["page"] += 1
        time.sleep(0.3)

    return pd.DataFrame(records)


def fetch_gbif_butterflies(min_x, min_y, max_x, max_y, max_records=2000):
    """Fetches butterfly occurrences from GBIF API using bounding box."""
    print("Fetching from GBIF...")
    records = []
    limit = 300
    offset = 0

    while len(records) < max_records:
        res = gbif_occ.search(
            taxonKey=7016,  # Superfamily Papilionoidea in GBIF
            decimalLatitude=f"{min_y},{max_y}",
            decimalLongitude=f"{min_x},{max_x}",
            hasCoordinate=True,
            limit=limit,
            offset=offset,
        )

        results = res.get("results", [])
        if not results:
            break

        for item in results:
            # Prefer species level scientific name
            species = item.get("species") or item.get("scientificName")
            lat = item.get("decimalLatitude")
            lng = item.get("decimalLongitude")

            if species and lat is not None and lng is not None:
                records.append({
                    "longitude": float(lng),
                    "latitude": float(lat),
                    "species_name": species,
                    "source": "GBIF",
                })

        if res.get("endOfRecords", True):
            break

        offset += limit
        time.sleep(0.2)

    return pd.DataFrame(records)


def extract_combined_butterflies(shapefile_path: str, output_csv: str = "combined_butterflies.csv"):
    # 1. Load Shapefile and transform CRS to WGS84 (EPSG:4326)
    print("Loading shapefile...")
    gdf = gpd.read_file(shapefile_path)
    if gdf.crs is None or gdf.crs.to_epsg() != 4326:
        gdf = gdf.to_crs(epsg=4326)

    try:
        aoi_geometry = gdf.geometry.union_all()
    except AttributeError:
        aoi_geometry = gdf.geometry.unary_union

    min_x, min_y, max_x, max_y = gdf.total_bounds

    # 2. Fetch data from both platforms
    df_inat = fetch_inaturalist_butterflies(min_x, min_y, max_x, max_y)
    df_gbif = fetch_gbif_butterflies(min_x, min_y, max_x, max_y)

    print(f"Retrieved: {len(df_inat)} records from iNaturalist, {len(df_gbif)} from GBIF.")

    # 3. Combine DataFrames
    combined_df = pd.concat([df_inat, df_gbif], ignore_index=True)
    if combined_df.empty:
        print("No butterfly observations found in the specified area.")
        return

    # 4. Remove spatial/species duplicates (GBIF often indexes iNaturalist data)
    # Round coordinates to ~11 meters precision to catch exact duplicate observations
    combined_df["lat_round"] = combined_df["latitude"].round(4)
    combined_df["lng_round"] = combined_df["longitude"].round(4)
    combined_df = combined_df.drop_duplicates(subset=["species_name", "lat_round", "lng_round"])
    combined_df = combined_df.drop(columns=["lat_round", "lng_round"])

    # 5. Point-in-Polygon spatial filter using the exact AOI geometry
    print("Filtering points strictly inside AOI polygon boundary...")
    obs_gdf = gpd.GeoDataFrame(
        combined_df,
        geometry=gpd.points_from_xy(combined_df.longitude, combined_df.latitude),
        crs="EPSG:4326",
    )

    inside_aoi = obs_gdf[obs_gdf.geometry.within(aoi_geometry)]

    # 6. Format and Save to CSV
    output_df = inside_aoi[["longitude", "latitude", "species_name", "source"]]
    output_df.to_csv(output_csv, index=False)
    print(f"Success! Saved {len(output_df)} unique records to '{output_csv}'.")


# --- Execution ---
if __name__ == "__main__":
    SHAPEFILE_PATH = "./shapefiles/kakamega_area_001.shp"
    extract_combined_butterflies(SHAPEFILE_PATH, "butterflies_aoi_combined.csv")

Loading shapefile...
Fetching from iNaturalist...
Fetching from GBIF...
Retrieved: 1267 records from iNaturalist, 7 from GBIF.
Filtering points strictly inside AOI polygon boundary...
Success! Saved 1161 unique records to 'butterflies_aoi_combined.csv'.


In [9]:
pd.read_csv("butterflies_aoi_combined.csv").head(10).columns

Index(['longitude', 'latitude', 'species_name', 'source'], dtype='str')

### Perform spatial thinning

In [ ]:
import pandas as pd
from pathlib import Path
import sys

# Shared_Functions root directory
project_root = Path.cwd().parent

sys.path.insert(0, str(project_root))

from occurrence.spatial_thin import spatial_thin


CSV_PATH = "./butterflies_aoi_combined.csv" 

insect_data = pd.read_csv(CSV_PATH)
# insect_data.head(10).columns

input_file = CSV_PATH
thinning_distance_m = 500  # 1 km
output_folder = "./shapefiles"   

thinned_occurrence_data = spatial_thin( input_file, thinning_distance_m, output_folder, lon_col=None, lat_col=None, seed=42)
thinned_occurrence_data.to_csv(f"butterflies_aoi_combined_thinned_{thinning_distance_m}.csv", index=False)

Loading CSV...
Records after duplicate removal: 688
Using EPSG:32636
Building KDTree for 688 records...


/home/vincent/Development/geospatial-analysis/Biodiversity-Project/Shared_Functions/occurrence/spatial_thin.py:206: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  thinned.to_file(shp_file)
/home/vincent/miniforge3/envs/geo_env/lib/python3.12/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'species_name' to 'species_na'
  ogr_write(


Creating map...

SPATIAL THINNING COMPLETE
Original records          : 1,161
Duplicates removed        : 473
Records before thinning   : 688
Records after thinning    : 155
Records removed by thinning: 533
Thinning distance         : 500 m

Outputs:
./shapefiles/occurrences_thinned_500m.csv
./shapefiles/occurrences_thinned_500m.shp
./shapefiles/occurrences_thinned_500m.gpkg
./shapefiles/spatial_thinning_500m.png


In [9]:
thinned_occurrence_data.head(10)

,longitude,latitude,species_name,source,geometry,lon,lat
0,34.753838,-0.090328,Hesperiinae,iNaturalist,POINT (34.75384 -0.09033),34.753838,-0.090328
1,34.884008,0.227750,Amauris echeria,iNaturalist,POINT (34.88401 0.22775),34.884008,0.227750
2,34.886391,0.199488,Hypolimnas dinarcha grandis,iNaturalist,POINT (34.88639 0.19949),34.886391,0.199488
3,34.747307,-0.112472,Junonia terea,iNaturalist,POINT (34.74731 -0.11247),34.747307,-0.112472
4,34.880631,0.255308,Hypolimnas dinarcha,iNaturalist,POINT (34.88063 0.25531),34.880631,0.255308
5,34.866452,0.238867,Evena crithea,iNaturalist,POINT (34.86645 0.23887),34.866452,0.238867
6,34.868723,0.299333,Belenois aurota aurota,iNaturalist,POINT (34.86872 0.29933),34.868723,0.299333
8,34.865808,0.347901,Evena crithea,iNaturalist,POINT (34.86581 0.3479),34.865808,0.347901
10,34.856133,0.291683,Euphaedrana,iNaturalist,POINT (34.85613 0.29168),34.856133,0.291683
11,34.895421,0.230763,Cymothoe hobarti hobarti,iNaturalist,POINT (34.89542 0.23076),34.895421,0.230763


### Clip ratsers to AOI

In [29]:

from raster.clip_raster_to_aoi import clip_rasters_to_aoi
clip_rasters_to_aoi(
    raster_folder="/run/media/vincent/Extreme Pro/Data/variables/Kenya_1km/Resubmit/Predictor_Variables_250",
    aoi_shapefile="shapefiles/kakamega_area_001.shp",
    output_folder="/run/media/vincent/Extreme Pro/Data/variables/Kenya_1km/Resubmit/Kakamega_Predictor_Variables_250"
)

Loading AOI...
Found 7 rasters

Processing: CanopyHeight.tif
✓ Saved: /run/media/vincent/Extreme Pro/Data/variables/Kenya_1km/Resubmit/Kakamega_Predictor_Variables_250/CanopyHeight.tif

Processing: HumanImpact.tif
✓ Saved: /run/media/vincent/Extreme Pro/Data/variables/Kenya_1km/Resubmit/Kakamega_Predictor_Variables_250/HumanImpact.tif

Processing: Precipitation.tif
✓ Saved: /run/media/vincent/Extreme Pro/Data/variables/Kenya_1km/Resubmit/Kakamega_Predictor_Variables_250/Precipitation.tif

Processing: TCB.tif
✓ Saved: /run/media/vincent/Extreme Pro/Data/variables/Kenya_1km/Resubmit/Kakamega_Predictor_Variables_250/TCB.tif

Processing: TCG.tif
✓ Saved: /run/media/vincent/Extreme Pro/Data/variables/Kenya_1km/Resubmit/Kakamega_Predictor_Variables_250/TCG.tif

Processing: TCW.tif
✓ Saved: /run/media/vincent/Extreme Pro/Data/variables/Kenya_1km/Resubmit/Kakamega_Predictor_Variables_250/TCW.tif

Processing: Temperature.tif
✓ Saved: /run/media/vincent/Extreme Pro/Data/variables/Kenya_1km/Resub

In [23]:


import rasterio
from rasterio.features import geometry_mask
import geopandas as gpd
import numpy as np

raster = "/run/media/vincent/Extreme Pro/Data/variables/Kenya_1km/Resubmit/Kakamega_Predictor_Variables_250/CanopyHeight.tif"
aoi = gpd.read_file("/home/vincent/Development/geospatial-analysis/Biodiversity-Project/Shared_Functions/GeoMeadian/shapefiles/kakamega_area_001.shp")

with rasterio.open(raster) as src:

    if aoi.crs != src.crs:
        aoi = aoi.to_crs(src.crs)

    mask_arr = geometry_mask(
        aoi.geometry,
        out_shape=(src.height, src.width),
        transform=src.transform,
        invert=True
    )

    data = src.read(1)

    outside_pixels = data[~mask_arr]

    print("Outside min:", outside_pixels.min())
    print("Outside max:", outside_pixels.max())
    print("Unique sample:", np.unique(outside_pixels[:1000]))

Outside min: 0
Outside max: 0
Unique sample: [0]


In [24]:
import rasterio

with rasterio.open("/run/media/vincent/Extreme Pro/Data/variables/Kenya_1km/Resubmit/Kakamega_Predictor_Variables_250/CanopyHeight.tif") as src:
    print("NoData:", src.nodata)

NoData: None


In [35]:
import os
from pathlib import Path
import rioxarray
from rasterio.enums import Resampling

# 1. Define input directory and output directory
input_dir = Path("/run/media/vincent/Extreme Pro/Data/variables/Kenya_1km/Resubmit/Kakamega_Predictor_Variables_250")
output_dir = input_dir / "aligned"
output_dir.mkdir(parents=True, exist_ok=True)

# 2. Gather all GeoTIFF files
raster_files = sorted(list(input_dir.glob("*.tif")) + list(input_dir.glob("*.asc")))

if not raster_files:
    raise FileNotFoundError(f"No .tif or .asc rasters found in {input_dir}")

# 3. Load reference raster template
ref_file = raster_files[0]
print(f"Reference Template: {ref_file.name}")
ref_ds = rioxarray.open_rasterio(ref_file)

# 4. Align all rasters to match reference template
for r_file in raster_files:
    out_file = output_dir / r_file.name
    
    with rioxarray.open_rasterio(r_file) as src:
        # Reproject & match grid geometry, resolution, and extent
        # Use Resampling.bilinear for continuous data (temp, precip, elevation)
        # Use Resampling.nearest for discrete/categorical data (land cover)
        aligned_ds = src.rio.reproject_match(
            ref_ds, 
            resampling=Resampling.bilinear
        )
        
        # Ensure nodata values are preserved
        if src.rio.nodata is not None:
            aligned_ds.rio.write_nodata(src.rio.nodata, inplace=True)
            
        # Write aligned raster to disk
        aligned_ds.rio.to_raster(out_file)
        
    print(f"Aligned: {r_file.name} -> Shape: {aligned_ds.shape[1:]}")

print(f"\nAll aligned rasters successfully saved to:\n{output_dir}")

Reference Template: CanopyHeight.tif
Aligned: CanopyHeight.tif -> Shape: (327, 316)
Aligned: HumanImpact.tif -> Shape: (327, 316)
Aligned: Precipitation.tif -> Shape: (327, 316)
Aligned: TCB.tif -> Shape: (327, 316)
Aligned: TCG.tif -> Shape: (327, 316)
Aligned: TCW.tif -> Shape: (327, 316)
Aligned: Temperature.tif -> Shape: (327, 316)

All aligned rasters successfully saved to:
/run/media/vincent/Extreme Pro/Data/variables/Kenya_1km/Resubmit/Kakamega_Predictor_Variables_250/aligned


In [37]:
from pathlib import Path
import glob
import time
import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.windows import Window
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import RandomizedSearchCV, StratifiedKFold
from shapely.geometry import Point

# ==========================================
# 1. CONFIGURATION & PATHS
# ==========================================
CSV_PATH = "./butterflies_aoi_combined_thinned_500.csv"  # Path to your combined GBIF & iNaturalist CSV
# RASTER_DIR = "/home/vincent/Development/geospatial-analysis/Biodiversity-Project/GeoMeadian/outputs/geomad_indices"
RASTER_DIR = "/home/vincent/Development/geospatial-analysis/Biodiversity-Project/Shared_Functions/GeoMeadian/outputs/geomad_indices"  # Path to your predictor rasters
# RASTER_DIR = "/run/media/vincent/Extreme Pro/Data/variables/Kenya_1km/Resubmit/Kakamega_Predictor_Variables_250/aligned"  # Path to your predictor rasters

OUTPUT_SUITABILITY_RASTER = "habitat_suitability_map.tif"

N_ITER_SEARCH = 15  # Iterations for hyperparameter tuning
RANDOM_STATE = 42


# ==========================================
# 2. HELPER FUNCTIONS
# ==========================================
def get_raster_files(raster_folder: str):
    """Finds and sorts all GeoTIFF predictor files in the folder."""
    files = sorted(glob.glob(f"{raster_folder}/*.tif") + glob.glob(f"{raster_folder}/*.tiff"))
    if not files:
        raise FileNotFoundError(f"No GeoTIFF rasters found in: {raster_folder}")
    print(f"Found {len(files)} predictor rasters:")
    for f in files:
        print(f"  - {Path(f).stem}")
    return files


def extract_raster_values_at_points(points_gdf: gpd.GeoDataFrame, raster_paths: list, feature_names: list):
    """Extracts values from multiple raster files at vector point locations."""
    extracted_data = []
    
    for raster_path, feat_name in zip(raster_paths, feature_names):
        with rasterio.open(raster_path) as src:
            # Match CRS if different
            if points_gdf.crs != src.crs:
                points_gdf = points_gdf.to_crs(src.crs)
            
            coords = [(geom.x, geom.y) for geom in points_gdf.geometry]
            sampled_vals = [val[0] for val in src.sample(coords)]
            
            # Mask out NoData values
            nodata = src.nodata
            if nodata is not None:
                sampled_vals = [np.nan if (v == nodata or np.isnan(v)) else v for v in sampled_vals]
                
            extracted_data.append(pd.Series(sampled_vals, name=feat_name))
            
    return pd.concat(extracted_data, axis=1), points_gdf.crs


def generate_pseudo_absences(num_points: int, raster_paths: list, crs):
    """
    Generates pseudo-absences strictly on pixels that are valid across 
    ALL predictor rasters (Composite Valid Mask).
    """
    print(f"\nBuilding composite valid mask across all {len(raster_paths)} rasters...")
    composite_valid_mask = None
    
    with rasterio.open(raster_paths[0]) as template:
        for path in raster_paths:
            with rasterio.open(path) as src:
                data = src.read(1)
                nodata = src.nodata
                
                if nodata is not None:
                    valid = (data != nodata) & (~np.isnan(data))
                else:
                    valid = ~np.isnan(data)
                    
                if composite_valid_mask is None:
                    composite_valid_mask = valid
                else:
                    composite_valid_mask &= valid  # Logical AND across all layers

        valid_indices = np.argwhere(composite_valid_mask)
        total_valid_pixels = len(valid_indices)
        
        if total_valid_pixels < num_points:
            print(f"Warning: Requested {num_points} points, but only {total_valid_pixels} valid pixels exist.")
            num_points = total_valid_pixels
            
        print(f"Sampling {num_points} background points from {total_valid_pixels:,} available pixels...")
        rng = np.random.default_rng(RANDOM_STATE)
        chosen_idx = rng.choice(total_valid_pixels, size=num_points, replace=False)
        selected_pixels = valid_indices[chosen_idx]
        
        # Map pixel coordinates (row, col) to spatial coordinates (x, y)
        points = [Point(template.xy(r, c)) for r, c in selected_pixels]
        
        # Construct GeoDataFrame without inline scalar assignment to prevent ValueError
        pseudo_gdf = gpd.GeoDataFrame(geometry=points, crs=crs)
        pseudo_gdf["target"] = 0
        
        return pseudo_gdf


# ==========================================
# 3. MAIN WORKFLOW
# ==========================================
def run_habitat_suitability_pipeline():
    start_time = time.time()
    
    # --- Step A: Load Rasters & Occurrences ---
    raster_files = get_raster_files(RASTER_DIR)
    feature_names = [Path(f).stem for f in raster_files]
    
    print("\nLoading occurrence CSV...")
    df_occ = pd.read_csv(CSV_PATH)
    
    presence_gdf = gpd.GeoDataFrame(
        df_occ,
        geometry=gpd.points_from_xy(df_occ.longitude, df_occ.latitude),
        crs="EPSG:4326"
    )
    presence_gdf["target"] = 1
    
    # --- Step B: Extract Predictors for Presence Points ---
    print("\nExtracting predictor values for presence points...")
    presence_env, raster_crs = extract_raster_values_at_points(presence_gdf, raster_files, feature_names)
    presence_df = pd.concat([presence_gdf[["target"]], presence_env], axis=1).dropna().reset_index(drop=True)
    
    num_presences = len(presence_df)
    print(f"Valid presence samples (after removing NoData): {num_presences}")
    
    # --- Step C: Generate & Extract 1:1 Pseudo-Absences ---
    pseudo_gdf = generate_pseudo_absences(num_presences, raster_files, raster_crs)
    absence_env, _ = extract_raster_values_at_points(pseudo_gdf, raster_files, feature_names)
    absence_df = pd.concat([pseudo_gdf[["target"]], absence_env], axis=1).dropna().reset_index(drop=True)
    
    # Ensure strict 1:1 ratio
    if len(absence_df) > num_presences:
        absence_df = absence_df.iloc[:num_presences]
        
    print(f"\nDataset Balance:")
    print(f"  - Presences (Class 1): {len(presence_df)}")
    print(f"  - Pseudo-Absences (Class 0): {len(absence_df)}")

    # --- Step D: Construct Machine Learning Dataset ---
    full_dataset = pd.concat([presence_df, absence_df], ignore_index=True)
    X = full_dataset[feature_names]
    y = full_dataset["target"]
    
    # --- Step E: Model Optimization (Random Forest) ---
    print("\nOptimizing Random Forest Classifier via RandomizedSearchCV...")
    rf_base = RandomForestClassifier(random_state=RANDOM_STATE, class_weight="balanced_subsample", oob_score=True)
    
    param_dist = {
        'n_estimators': [100, 200, 300, 500],
        'max_depth': [None, 10, 20, 30],
        'min_samples_split': [2, 5, 10],
        'min_samples_leaf': [1, 2, 4],
        'max_features': ['sqrt', 'log2', 0.5]
    }
    
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
    
    search = RandomizedSearchCV(
        estimator=rf_base,
        param_distributions=param_dist,
        n_iter=N_ITER_SEARCH,
        scoring='roc_auc',
        cv=cv,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=1
    )
    
    search.fit(X, y)
    best_rf = search.best_estimator_
    
    print(f"\n--- Model Performance ---")
    print(f"Best Parameters: {search.best_params_}")
    print(f"Best Cross-Validation ROC-AUC: {search.best_score_:.4f}")
    print(f"Out-of-Bag (OOB) Accuracy: {best_rf.oob_score_:.4f}")
    
    importances = pd.Series(best_rf.feature_importances_, index=feature_names).sort_values(ascending=False)
    print("\nFeature Importances:")
    print(importances.to_string())

    # --- Step F: Spatial Prediction across Raster Extent ---
    print("\nPredicting habitat suitability across raster spatial extent...")
    src_files = [rasterio.open(f) for f in raster_files]
    template_src = src_files[0]
    
    profile = template_src.profile.copy()
    profile.update({
        'dtype': 'float32',
        'count': 1,
        'nodata': -9999.0,
        'compress': 'lzw'
    })
    
    # Process tile-by-tile / block-by-block to conserve RAM
    with rasterio.open(OUTPUT_SUITABILITY_RASTER, 'w', **profile) as dst:
        for block_idx, window in template_src.block_windows(1):
            
            block_data = []
            is_nodata_mask = None
            
            for src in src_files:
                data = src.read(1, window=window)
                nodata = src.nodata
                
                if nodata is not None:
                    invalid = (data == nodata) | np.isnan(data)
                else:
                    invalid = np.isnan(data)
                    
                if is_nodata_mask is None:
                    is_nodata_mask = invalid
                else:
                    is_nodata_mask |= invalid
                    
                block_data.append(data.ravel())
                
            block_matrix = np.column_stack(block_data)
            valid_pixels_mask = ~is_nodata_mask.ravel()
            
            out_block = np.full(block_matrix.shape[0], -9999.0, dtype=np.float32)
            
            if np.any(valid_pixels_mask):
                valid_X = block_matrix[valid_pixels_mask]
                # Probability of Class 1 (Habitat Suitability)
                suitability_probs = best_rf.predict_proba(valid_X)[:, 1]
                out_block[valid_pixels_mask] = suitability_probs
                
            out_block_2d = out_block.reshape(template_src.read(1, window=window).shape)
            dst.write(out_block_2d, 1, window=window)

    # Close input raster handlers
    for src in src_files:
        src.close()
        
    print(f"\nFinished! Output suitability raster written to: {OUTPUT_SUITABILITY_RASTER}")
    print(f"Total Execution Time: {time.time() - start_time:.2f} seconds")


if __name__ == "__main__":
    run_habitat_suitability_pipeline()

Found 8 predictor rasters:
  - EVI
  - MSAVI
  - NBR
  - NBR2
  - NDMI
  - NDVI
  - NDWI
  - SAVI

Loading occurrence CSV...

Extracting predictor values for presence points...
Valid presence samples (after removing NoData): 155

Building composite valid mask across all 8 rasters...
Sampling 155 background points from 51,871,014 available pixels...

Dataset Balance:
  - Presences (Class 1): 155
  - Pseudo-Absences (Class 0): 155

Optimizing Random Forest Classifier via RandomizedSearchCV...
Fitting 5 folds for each of 15 candidates, totalling 75 fits

--- Model Performance ---
Best Parameters: {'n_estimators': 300, 'min_samples_split': 5, 'min_samples_leaf': 4, 'max_features': 'sqrt', 'max_depth': None}
Best Cross-Validation ROC-AUC: 0.8427
Out-of-Bag (OOB) Accuracy: 0.7774

Feature Importances:
NBR2     0.200885
NBR      0.168463
NDVI     0.137524
NDWI     0.122777
SAVI     0.097014
MSAVI    0.095599
NDMI     0.092333
EVI      0.085405

Predicting habitat suitability across raster spa

/home/vincent/miniforge3/envs/geo_env/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/home/vincent/miniforge3/envs/geo_env/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/home/vincent/miniforge3/envs/geo_env/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/home/vincent/miniforge3/envs/geo_env/lib/python3.12/site-packages/sklearn/utils/validation.py:2827: UserWarning: X does not have valid feature names, but RandomForestClassifier was fitted with feature names
  warnings.warn(
/home/vincent/miniforge3/envs/geo_env/lib/python3.12/site-packages/sklearn/utils/validation.py:2827:


Finished! Output suitability raster written to: habitat_suitability_map.tif
Total Execution Time: 345.85 seconds
